Vehicle Dataset Training

First of all, I will get the images per (interval)th frames from the footage in order to label or annotate them as per target images in my dataset.

In [2]:
import cv2, os

In [3]:
from ultralytics import YOLO

For interval of frame (Target images=200 because I want to train 200 images in my dataset)

In [ ]:
cap = cv2.VideoCapture(r"L:\CCTV\20250616\N02174326.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"FPS: {fps}, total frames: {total_frames}, duration: {total_frames/fps/60:.1f} min")

TARGET_IMAGES = 200          # <- set how many images you actually want
interval = max(1, total_frames // TARGET_IMAGES)
print(f"Will save every {interval}th frame → about {total_frames // interval} images")

cap.release()

FPS: 13.999992230858917, total frames: 25228, duration: 30.0 min
Will save every 126th frame → about 200 images


now I will input i % 126==0

In [4]:


os.makedirs(r"L:\Softwares\VS Code\Files\YOLO\vehicle_dataset\images", exist_ok=True)
cap = cv2.VideoCapture(r"L:\CCTV\20250616\N02174326.mp4")
i, saved = 0, 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    i += 1
    if i % 126 == 0:      # every 126th frame — adjust for how much footage you have
        cv2.imwrite(rf"L:\Softwares\VS Code\Files\YOLO\vehicle_dataset\images\frame_{saved:05d}.jpg", frame)
        saved += 1
cap.release()
print(f"Saved {saved} frames")

Saved 200 frames


Launch labelImg in Terminal for annotation of images.
Enter this code in terminal:
labelImg "L:\Softwares\VS Code Files\YOLO\vehicle_dataset\images" "L:\Softwares\VS Code Files\YOLO\vehicle_dataset\classes.txt"

Speed it up: pre-label with my current model first.
Rather than drawing every box from nothing, let yolo11n.pt draw its guesses first, so I am just correcting labels and boundaries in LabelImg rather than starting from a blank image. It only outputs 6 of your classes, but that still covers most boxes.

In [ ]:
model = YOLO("yolo11n.pt")
os.makedirs(r"L:\Softwares\VS Code\Files\YOLO\vehicle_dataset\labels", exist_ok=True)

coco_to_yours = {0: 2, 1: 1, 2: 3, 3: 0, 5: 9, 7: 7}  # person,bicycle,car,motorcycle,bus,truck -> your class ids

results = model.predict(r"L:\Softwares\VS Code\Files\YOLO\vehicle_dataset\images", stream=True, conf=0.3)
for r in results:
    name = os.path.splitext(os.path.basename(r.path))[0]
    lines = []
    for box in r.boxes:
        coco_id = int(box.cls[0])
        if coco_id not in coco_to_yours:
            continue
        cls_id = coco_to_yours[coco_id]
        x, y, w, h = box.xywhn[0].tolist()   # already normalized
        lines.append(f"{cls_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")
    with open(rf"L:\Softwares\VS Code\Files\YOLO\vehicle_dataset\labels\{name}.txt", "w") as f:
        f.write("\n".join(lines))

print("Pre-labels written. Open LabelImg to correct classes and add missing ones (van, tractor, etc.)")


image 1/200 L:\Softwares\VS Code\Files\YOLO\vehicle_dataset\images\frame_00000.jpg: 640x576 4 persons, 464.6ms
image 2/200 L:\Softwares\VS Code\Files\YOLO\vehicle_dataset\images\frame_00001.jpg: 640x576 6 persons, 2 bicycles, 206.4ms
image 3/200 L:\Softwares\VS Code\Files\YOLO\vehicle_dataset\images\frame_00002.jpg: 640x576 3 persons, 1 motorcycle, 203.5ms
image 4/200 L:\Softwares\VS Code\Files\YOLO\vehicle_dataset\images\frame_00003.jpg: 640x576 4 persons, 1 car, 1 bus, 198.4ms
image 5/200 L:\Softwares\VS Code\Files\YOLO\vehicle_dataset\images\frame_00004.jpg: 640x576 3 persons, 2 cars, 1 motorcycle, 217.0ms
image 6/200 L:\Softwares\VS Code\Files\YOLO\vehicle_dataset\images\frame_00005.jpg: 640x576 1 person, 1 truck, 206.1ms
image 7/200 L:\Softwares\VS Code\Files\YOLO\vehicle_dataset\images\frame_00006.jpg: 640x576 3 persons, 4 bicycles, 319.2ms
image 8/200 L:\Softwares\VS Code\Files\YOLO\vehicle_dataset\images\frame_00007.jpg: 640x576 3 persons, 3 motorcycles, 207.4ms
image 9/200 L:

Now, I will open labelImg from terminal

Once labeling is done, split into train/val and write data.yaml

Split

In [2]:
import random, shutil

random.seed(0)
imgs = os.listdir(r"L:\Softwares\VS Code\Files\YOLO\vehicle_dataset\images")
random.shuffle(imgs)
split = int(0.85 * len(imgs))

for folder, subset in [("train", imgs[:split]), ("val", imgs[split:])]:
    os.makedirs(rf"L:\Softwares\VS Code\Files\YOLO\vehicle_dataset\{folder}\images", exist_ok=True)
    os.makedirs(rf"L:\Softwares\VS Code\Files\YOLO\vehicle_dataset\{folder}\labels", exist_ok=True)
    for img in subset:
        name = os.path.splitext(img)[0]
        shutil.copy(rf"L:\Softwares\VS Code\Files\YOLO\vehicle_dataset\images\{img}", rf"L:\Softwares\VS Code\Files\YOLO\vehicle_dataset\{folder}\images\{img}")
        label = rf"L:\Softwares\VS Code\Files\YOLO\vehicle_dataset\labels\{name}.txt"
        if os.path.exists(label):
            shutil.copy(label, rf"L:\Softwares\VS Code\Files\YOLO\vehicle_dataset\{folder}\labels\{name}.txt")

Write data.yaml

In [3]:
yaml_content = """path: L:\Softwares\VS Code\Files\YOLO\vehicle_dataset
train: train/images
val: val/images

names:
  0: motorcycle
  1: bicycle
  2: pedestrian
  3: car
  4: van
  5: tractor
  6: tractor_with_trailer
  7: truck
  8: mini_truck
  9: bus
  10: mini_bus
  11: microbus
  12: utility_vehicle
  13: autorickshaw
"""

with open(r"L:\Softwares\VS Code\Files\YOLO\vehicle_dataset\data.yaml", "w") as f:
    f.write(yaml_content)

print("data.yaml written")

data.yaml written


<>:1: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<>:1: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
C:\Users\DELL\AppData\Local\Temp\ipykernel_7800\2612367246.py:1: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
  yaml_content = """path: L:\Softwares\VS Code\Files\YOLO\vehicle_dataset


Sanity check before training (Quickly confirm the split worked and nothing's empty)

In [4]:
BASE = r"L:\Softwares\VS Code\Files\YOLO\vehicle_dataset"

for folder in ["train", "val"]:
    img_dir = os.path.join(BASE, folder, "images")
    lbl_dir = os.path.join(BASE, folder, "labels")
    imgs = os.listdir(img_dir)
    labels = os.listdir(lbl_dir)
    print(f"{folder}: {len(imgs)} images, {len(labels)} labels")

    # flag images that ended up with no matching label file
    missing = [f for f in imgs if os.path.splitext(f)[0] + ".txt" not in labels]
    if missing:
        print(f"  WARNING: {len(missing)} images have no label file: {missing[:5]}{' ...' if len(missing) > 5 else ''}")

    # flag empty label files (image labeled but with zero boxes saved)
    empty = [f for f in labels if os.path.getsize(os.path.join(lbl_dir, f)) == 0]
    if empty:
        print(f"  NOTE: {len(empty)} label files are empty (no objects drawn): {empty[:5]}{' ...' if len(empty) > 5 else ''}")

train: 170 images, 170 labels
val: 30 images, 30 labels


Train the dataset

In [7]:
model = YOLO("yolo11n.pt")
model.train(data=r"L:\Softwares\VS Code\Files\YOLO\vehicle_dataset\data.yaml", epochs=10, imgsz=640)

New https://pypi.org/project/ultralytics/8.4.171 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.163  Python-3.14.3 torch-2.14.0+cpu CPU (Intel Core i5-4210U 1.70GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=L:\Softwares\VS Code\Files\YOLO\vehicle_dataset\data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  6,  7,  8,  9, 11, 12])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001FA28B30E60>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.

Training from CPU is too slow, so I would like to do training from Google Colab which uses GPU from cloud storage which is free. Training will be fast from it.

For this, I should check that PyTorch itself can see the GPU or not.

In [8]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: False
Device name: None


That means GPU is not available in my device and I must run colab to train from GPU.
I trained for 100 epoches in Colab (free GPU) which performed in only 12 mins

Now, download best.pt (output file from training) and used it instead of yolo11n.pt (Coco dataset pre-trained model)

I checked my self-trained best.pt in YOLO-1.ipynb file.